# Job 3 — Fixing the non-Minecraft weakness (AMRBART)
Job 1 showed the fine-tuned AMRBART is ~26 AnCast points worse on normal English (non-Minecraft) than on Minecraft lines (66.8 vs 92.6, split A). The paper names this as its main limitation (§4.2: non-Minecraft sentences are under-represented in training) but does not try to fix it.

**Fixes tested** (your choice: the two recommended methods + their combination), all on AMRBART with Job 1's training code:

| Run | What | Starts from | Train data |
|---|---|---|---|
| R1 Upsample | non-Minecraft sentences repeated 4× | AMRBART AMR parser (like Job 1) | split A train, non-Minecraft ×4 (16% → ~43%) |
| R2 Two-stage | continue training on non-Minecraft only | your Job 1 model (`amrbart_A/best`) | non-Minecraft part of split A train |
| R3 Combined | R1, then the R2 second stage | R1's best model | non-Minecraft part of split A train |

Then the **winner** (picked on **dev**, never on test) is re-run on clean split B.
**Significance:** paired bootstrap (1,000 resamples) of the SMATCH gain over Job 1's AMRBART on the same test sentences.

| Part | Time (A100) |
|---|---|
| 0 Setup · 1 Data/baseline | ~10 min |
| 2 R1, R2, R3 on split A (+ tests) | ~1.5–2 h |
| 3 Winner on split B | ~15 min – 1.2 h (depends on winner) |
| 4 Final table | ~5 min |

Use a **fresh runtime** (this job pins `transformers==4.46.3` like Job 1).

# Part 0 — Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/umr_job1'
WORK  = '/content/job3'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
print(os.listdir(f'{DRIVE}/runs'))

Mounted at /content/drive
/content/job3
['amrbart_A', 'amrbart_B', 'qwen3_A', 'qwen3_B', 'j3_upsample_A', 'j3_twostage_A', 'j3_combined_A', 'j3_winner_B']


Same installs and pinned versions as Job 1.

In [3]:
!pip -q install "transformers==4.46.3" penman smatch mip networkx regex
!pip -q install git+https://github.com/flipz357/smatchpp.git
!git clone -q https://github.com/umr4nlp/ancast.git && pip -q install -e ancast
!git clone -q https://github.com/goodbai-nlp/AMRBART.git && git -C AMRBART checkout -q 37e5742411e74d4269c33e8f2c5f5a8a0ed51241
import sys; sys.path.insert(0, WORK); sys.path.insert(0, f'{WORK}/ancast')
import warnings, logging; warnings.filterwarnings('ignore'); logging.getLogger('penman').setLevel(logging.ERROR)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.3/191.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requi

### Code files
- `umr_tools.py`, `setup_amrbart_tok.py`, `umr_tokenizer.py` — identical to Job 1.
- `train_amrbart.py` — Job 1's loop + one new option `select="balanced"` (model selection on Minecraft **and** non-Minecraft dev; Job 1's behaviour is unchanged when not used).
- `job3_tools.py` — upsampling + paired bootstrap.

In [4]:
%%writefile umr_tools.py
"""
umr_tools.py - data, split and evaluation helpers for replicating SETUP
(Markle, Gutierrez Bach & Wein, LREC 2026) with AMRBART.

Sections of the paper referenced below:
  Sec 3.3 Data, Sec 3.4 Data split, Sec 3.5 Metrics, Sec 4.1 Minecraft breakdown.
"""
import glob
import json
import os
import random
import re
import subprocess
import sys
from collections import Counter, defaultdict

import logging
import penman
logging.getLogger("penman").setLevel(logging.ERROR)

BLOCK_SEP = re.compile(r"^#{20,}\s*$", re.M)
MINECRAFT_RE = re.compile(r"\[Builder|<Builder>|<Architect>|\[Architect")
BACKOFF_GRAPH = "(z0 / empty-graph)"   # used when a prediction cannot be parsed


# ---------------------------------------------------------------- reading
def _section(block, start, ends):
    """Text between a '# start' header and the next '# end' header."""
    i = block.find(start)
    if i < 0:
        return ""
    txt = block[i + len(start):]
    for e in ends:
        j = txt.find(e)
        if j >= 0:
            txt = txt[:j]
    return txt.strip()


def read_umr_file(path):
    """Yield one record per sentence block of a .umr file."""
    raw = open(path, encoding="utf-8").read()
    doc = os.path.basename(path)
    for k, block in enumerate(b for b in BLOCK_SEP.split(raw) if b.strip()):
        m = re.search(r"^Words:[ \t]*(.*)$", block, re.M)
        if m:
            sent = " ".join(m.group(1).split())
        else:  # some files only have '# :: snt1  text'
            m = re.search(r"^#[ \t]*::[ \t]*snt\d+[ \t]*(.*)$", block, re.M)
            sent = " ".join(m.group(1).split()) if m else ""
        graph = _section(block, "# sentence level graph:",
                         ["# alignment:", "# document level annotation:"])
        meta = re.search(r"sent_id\s*=\s*(\S+)", block)
        yield {"doc": doc, "idx": k, "sent_id": meta.group(1) if meta else f"{doc}#{k}",
               "sentence": sent, "graph_raw": graph}


def balance_parens(s):
    """Fix the most common annotation error: too many / too few ')' (parens inside quotes ignored)."""
    depth, out, in_q = 0, [], False
    for ch in s:
        if ch == '"':
            in_q = not in_q
        elif not in_q and ch == "(":
            depth += 1
        elif not in_q and ch == ")":
            if depth == 0:          # extra closing paren -> drop it
                continue
            depth -= 1
        out.append(ch)
    return "".join(out).rstrip() + ")" * depth


def clean_graph(s):
    """Return a canonical PENMAN string, or None if the graph is empty/unparseable."""
    s = (s or "").strip()
    if not s or s.replace(" ", "") in ("()", "(())"):
        return None
    try:
        g = penman.decode(balance_parens(s))
        if len(g.instances()) == 0:
            return None
        g.metadata = {}
        return penman.encode(g, indent=4)
    except Exception:
        return None


def is_minecraft(sentence):
    """Sec 4.1: Minecraft = sentence contains a Builder/Architect tag."""
    return bool(MINECRAFT_RE.search(sentence))


def load_english(umr_dir):
    """Read every English .umr file, clean graphs, return (records, stats)."""
    files = sorted(f for f in glob.glob(os.path.join(umr_dir, "**", "*.umr"), recursive=True)
                   if "english" in f.lower())
    if not files:
        raise FileNotFoundError(f"no English .umr files under {umr_dir}")
    recs, stats = [], Counter()
    for f in files:
        for r in read_umr_file(f):
            stats["blocks"] += 1
            if not r["sentence"]:
                stats["no_sentence"] += 1
                continue
            raw_ok = True
            try:
                penman.decode(r["graph_raw"])
            except Exception:
                raw_ok = False
            g = clean_graph(r["graph_raw"])
            if g is None:
                stats["empty_or_broken_graph"] += 1
                continue
            if not raw_ok:
                stats["fixed_by_paren_balancing"] += 1
            r["umr"] = g
            r["is_minecraft"] = is_minecraft(r["sentence"])
            del r["graph_raw"]
            recs.append(r)
    stats["files"] = len(files)
    stats["kept"] = len(recs)
    stats["minecraft"] = sum(r["is_minecraft"] for r in recs)
    return recs, stats


# ---------------------------------------------------------------- splits
def read_gold_sentences(path):
    """Sentences (# ::snt lines) of an AMR-style file, e.g. the authors' dev gold file."""
    return [" ".join(l[len("# ::snt"):].split())
            for l in open(path, encoding="utf-8") if l.startswith("# ::snt")]


def split_paper_style(recs, author_dev_sents, test_size=2941, seed=42):
    """Split A (Sec 3.4 replication): the authors' exact dev set + random sentence-level test."""
    need = Counter(author_dev_sents)
    dev, rest = [], []
    for r in recs:
        if need[r["sentence"]] > 0:
            need[r["sentence"]] -= 1
            dev.append(r)
        else:
            rest.append(r)
    rng = random.Random(seed)
    rng.shuffle(rest)
    test, train = rest[:test_size], rest[test_size:]
    unmatched = sum(need.values())
    return {"train": train, "dev": dev, "test": test}, unmatched


def split_clean(recs, dev_frac=0.1, test_frac=0.1, seed=42):
    """Split B (ours): split by DOCUMENT, then drop dev/test sentences whose exact
    text also occurs in train (removes the repeated Minecraft lines)."""
    by_doc = defaultdict(list)
    for r in recs:
        by_doc[r["doc"]].append(r)
    docs = sorted(by_doc)
    random.Random(seed).shuffle(docs)
    n = len(recs)
    parts, acc, cur = {"test": [], "dev": [], "train": []}, 0, "test"
    for d in docs:
        if cur == "test" and acc >= test_frac * n:
            cur, acc = "dev", 0
        if cur == "dev" and acc >= dev_frac * n:
            cur = "train"
        parts[cur].extend(by_doc[d])
        acc += len(by_doc[d])
    train_sents = {r["sentence"] for r in parts["train"]}
    removed = {}
    for k in ("dev", "test"):
        before = len(parts[k])
        parts[k] = [r for r in parts[k] if r["sentence"] not in train_sents]
        removed[k] = before - len(parts[k])
    return parts, removed


def mask_numbers(s):
    return re.sub(r"-?\d+", "N", s)


def leakage_report(split):
    """How many test sentences already appear in train (exactly / up to numbers)."""
    tr = {r["sentence"] for r in split["train"]}
    trm = {mask_numbers(s) for s in tr}
    out = {}
    for k in ("dev", "test"):
        xs = split[k]
        if not xs:
            continue
        ex = sum(r["sentence"] in tr for r in xs)
        tm = sum(mask_numbers(r["sentence"]) in trm for r in xs)
        out[k] = {"n": len(xs), "exact_in_train_%": round(100 * ex / len(xs), 1),
                  "template_in_train_%": round(100 * tm / len(xs), 1)}
    return out


def save_jsonl(recs, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


# ---------------------------------------------------------------- linearization
def _tokenize_encoded_graph(encoded):
    lin = re.sub(r"(\".+?\")", r" \1 ", encoded)
    pieces = []
    for p in lin.split():
        if p.startswith('"') and p.endswith('"'):
            pieces.append(p)
        else:
            for a, b in (("(", " ( "), (")", " ) "), (":", " :"), ("/", " / ")):
                p = p.replace(a, b)
            pieces.append(p.strip())
    return re.sub(r"\s+", " ", " ".join(pieces)).strip().split(" ")


def linearize(penman_str):
    """DFS linearization with <pointer:i> variables - same scheme AMRBART was trained on
    (AMR-Process `dfs_linearize`), so the pretrained parser sees a familiar format."""
    g = penman.decode(penman_str)
    g.metadata = {}
    toks = _tokenize_encoded_graph(penman.encode(g))
    remap = {}
    for i in range(1, len(toks)):
        if toks[i] == "/" and toks[i - 1] not in remap:
            remap[toks[i - 1]] = f"<pointer:{len(remap)}>"
    out, i = [toks[0]], 1
    while i < len(toks):
        nxt, lst = toks[i], out[-1]
        if nxt in remap:
            if lst == "(" and i + 1 < len(toks) and toks[i + 1] == "/":
                nxt = remap[nxt]
                i += 1
            elif lst.startswith(":"):
                nxt = remap[nxt]
        out.append(nxt)
        i += 1
    return out


# ---------------------------------------------------------------- evaluation
def to_valid(graph_str):
    """Parseable graph or the backoff graph, so no sentence is silently skipped by a scorer.
    A graph counts as valid only if penman AND smatch's own parser both accept it and
    every node has a concept."""
    try:
        g = penman.decode(graph_str)
        # drop empty values like :op1 "" (they carry no meaning and crash AnCast)
        g = penman.Graph([t for t in g.triples if t[2] not in ('""', "")], top=g.top)
        if not g.instances() or any(i.target in (None, "") for i in g.instances()):
            raise ValueError("node without concept")
        g.metadata = {}
        s = penman.encode(g, indent=4)
        import io, contextlib, amr as smatch_amr          # smatch's parser
        with contextlib.redirect_stderr(io.StringIO()):
            if smatch_amr.AMR.parse_AMR_line(" ".join(s.split())) is None:
                raise ValueError("smatch cannot parse")
        return s, True
    except Exception:
        return BACKOFF_GRAPH, False


def write_amr_style(sents, graphs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs)):
            f.write(f"# ::id {i}\n# ::snt {s}\n{g}\n\n")


def score_smatch(pred_path, gold_path):
    import smatch
    with open(pred_path, encoding="utf-8") as p, open(gold_path, encoding="utf-8") as g:
        P, R, F = next(smatch.score_amr_pairs(p, g))
    return 100 * F


def score_smatchpp(pred_path, gold_path):
    """SMATCH++ with optimal (ILP) alignment, generic standardization, micro F1."""
    cmd = [sys.executable, "-m", "smatchpp", "-a", pred_path, "-b", gold_path,
           "-solver", "ilp", "-graph_type", "generic",
           "-score_dimension", "main", "-score_type", "micro"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.search(r"main ---->\s+F1:\s*([\d.]+)", out)
    return float(m.group(1)) if m else float("nan"), out


def write_umr_style(sents, graphs, path):
    """UMR block format that AnCast expects with data_format='umr'."""
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs), 1):
            f.write(f"# :: snt{i}\t{s}\n\n# sentence level graph:\n{g}\n\n"
                    f"# alignment:\n\n# document level annotation:\n\n\n")


def score_ancast(pred_path, gold_path, data_format="umr"):
    """AnCast (sentence scope). data_format='umr' scores UMR attributes such as
    :aspect / :refer-number; data_format='amr' silently ignores them (checked by us)."""
    from ancast import evaluate
    import logging
    logging.getLogger("ancast").setLevel(logging.ERROR)
    # pass FILE PATHS: AnCast only splits files into sentences; a raw string is
    # treated as one block and only its first sentence gets scored.
    res = evaluate(pred_path, gold_path, scope="snt", data_format=data_format,
                   output_csv_as_string=True)
    f = res[0] if isinstance(res, tuple) else res
    while isinstance(f, (list, tuple)):
        f = f[0]
    return 100 * float(f)


def evaluate_all(sents, preds, golds, flags, out_dir, name="test", run_smatchpp=True):
    """Scores for all / Minecraft / non-Minecraft (Sec 3.5 metrics + Sec 4.1 breakdown)."""
    os.makedirs(out_dir, exist_ok=True)
    fixed = [to_valid(p) for p in preds]
    preds_v = [g for g, _ in fixed]
    n_invalid = sum(not ok for _, ok in fixed)
    rows = {}
    for subset, keep in (("all", lambda f: True), ("minecraft", lambda f: f),
                         ("non_minecraft", lambda f: not f)):
        idx = [i for i, f in enumerate(flags) if keep(f)]
        if not idx:
            continue
        pp = os.path.join(out_dir, f"{name}_{subset}.pred.txt")
        gp = os.path.join(out_dir, f"{name}_{subset}.gold.txt")
        write_amr_style([sents[i] for i in idx], [preds_v[i] for i in idx], pp)
        write_amr_style([sents[i] for i in idx], [golds[i] for i in idx], gp)
        up, ug = pp.replace(".txt", ".umrfmt"), gp.replace(".txt", ".umrfmt")
        write_umr_style([sents[i] for i in idx], [preds_v[i] for i in idx], up)
        write_umr_style([sents[i] for i in idx], [golds[i] for i in idx], ug)
        row = {"n": len(idx), "AnCast": score_ancast(up, ug, "umr"),
               "AnCast_amrfmt": score_ancast(pp, gp, "amr"), "SMATCH": score_smatch(pp, gp)}
        if run_smatchpp:
            row["SMATCH++"] = score_smatchpp(pp, gp)[0]
        rows[subset] = {k: round(v, 2) if isinstance(v, float) else v for k, v in row.items()}
    rows["invalid_predictions"] = n_invalid
    with open(os.path.join(out_dir, f"{name}_scores.json"), "w") as f:
        json.dump(rows, f, indent=2)
    return rows

Writing umr_tools.py


In [5]:
%%writefile setup_amrbart_tok.py
"""
Copies AMRBART's graph tokenizer + post-processing into ./amrbart_tok/ and fixes
imports so it works with current `transformers`.
Source: github.com/goodbai-nlp/AMRBART (MIT license), fine-tune/ folder.
Usage:  python setup_amrbart_tok.py /path/to/AMRBART
"""
import os
import re
import shutil
import sys

src = os.path.join(sys.argv[1], "fine-tune")
dst = "amrbart_tok"
os.makedirs(dst, exist_ok=True)
for f in ["common/postprocessing.py", "common/penman_interface.py",
          "common/additional-tokens.json", "model_interface/tokenization_bart.py"]:
    shutil.copy(os.path.join(src, f), os.path.join(dst, os.path.basename(f)))

# constant.py imports optimizers removed from new transformers -> keep only what the tokenizer needs
orig = open(os.path.join(src, "common/constant.py"), encoding="utf-8").read()
recats = re.search(r"recategorizations = \[.*?\]", orig, re.S).group(0)
open(os.path.join(dst, "constant.py"), "w").write(
    "import os, json\n"
    "raw_special_tokens = json.load(open(os.path.join(os.path.dirname(__file__), "
    "'additional-tokens.json'), encoding='utf-8'))\n" + recats + "\n")

for f in ["postprocessing.py", "tokenization_bart.py"]:
    p = os.path.join(dst, f)
    t = open(p, encoding="utf-8").read()
    t = t.replace("from common.penman_interface import", "from .penman_interface import")
    t = t.replace("from common import postprocessing", "from . import postprocessing")
    t = t.replace("from common.constant import", "from .constant import")
    open(p, "w", encoding="utf-8").write(t)

open(os.path.join(dst, "__init__.py"), "w").write(
    "from .tokenization_bart import AMRBartTokenizer\nfrom . import postprocessing\n")
print("amrbart_tok ready:", sorted(os.listdir(dst)))

Writing setup_amrbart_tok.py


In [6]:
%%writefile umr_tokenizer.py
"""
UMRBartTokenizer: AMRBART's graph tokenizer with one fix for UMR.

Problem: UMR roles that are not in AMRBART's AMR vocabulary (e.g. :refer-number,
:aspect, :modal-strength) are split as [':', ' refer', '-', 'number']. The leading
space on ' refer' makes AMRBART's decoder start a NEW token, so ':' and 'refer-number'
come apart and the graph is corrupted (the attribute is dropped or mangled).
Fix: encode the role name without the leading space so the decoder glues it back to ':'.
"""
import regex as re
from amrbart_tok import AMRBartTokenizer


class UMRBartTokenizer(AMRBartTokenizer):
    def _bpe_nospace(self, text):
        out = []
        for piece in self.patterns.findall(text):
            piece = "".join(self.byte_encoder[b] for b in piece.encode("utf-8"))
            out.extend(self.bpe(piece).split(" "))
        return out

    def tokenize_amr(self, amr_tokens):
        ids = []
        for tokk in amr_tokens:
            is_rel = tokk.startswith(":") and len(tokk) > 1
            if is_rel and (self.INIT + tokk) not in self.encoder:
                if tokk.endswith("-of") and (self.INIT + tokk[:-3]) in self.encoder:
                    pieces = [self.INIT + tokk[:-3], "-of"]          # same as AMRBART
                elif tokk.endswith("-of"):
                    pieces = [self.INIT + ":"] + self._bpe_nospace(tokk[1:-3]) + ["-of"]
                else:
                    pieces = [self.INIT + ":"] + self._bpe_nospace(tokk[1:])
                ids.extend(self.encoder.get(p, self.unk_token_id) for p in pieces)
            else:
                ids.extend(super().tokenize_amr([tokk]))
        return ids


def decode_graph(tok, ids):
    """Our own ids -> PENMAN decoder (replaces AMRBART's AMR-specific post-processing,
    which drops UMR word-valued attributes such as ':aspect state' or ':refer-number plural').
    Returns (penman_string, ok)."""
    import penman
    from umr_tools import balance_parens
    stop = {tok.amr_eos_token_id, tok.eos_token_id}
    skip = {tok.amr_bos_token_id, tok.bos_token_id, tok.pad_token_id}
    words = []
    for i in ids:
        i = int(i)
        if i in stop:
            break
        if i in skip or i < 0:
            continue
        p = tok.decoder.get(i, "")
        # a number right after a complete role (":quant" + "33") is a new word even if the
        # model emitted it without the leading-space marker - otherwise we get ":quant33"
        after_role = bool(words) and words[-1].startswith(tok.INIT + ":") and words[-1] in tok.encoder
        if p.startswith(tok.INIT) or not words or (after_role and p[:1].isdigit()):
            words.append(p if p.startswith(tok.INIT) else tok.INIT + p)
        else:
            words[-1] += p
    words = [tok.convert_tokens_to_string([w]).strip() for w in words]
    out, in_lit, lit = [], False, []
    for k, w in enumerate(words):
        if w == "<lit>":
            in_lit, lit = True, []
        elif w == "</lit>":
            out.append('"' + " ".join(lit).replace('"', "") + '"')
            in_lit = False
        elif in_lit:
            lit.append(w)
        elif w.startswith("<pointer:") and w.endswith(">"):
            var = "z" + w[9:-1]
            out.append(var + " /" if out and out[-1] == "(" else var)
        elif w:
            out.append(w)
    if in_lit:
        out.append('"' + " ".join(lit) + '"')
    text = " ".join(out).replace("( ", "(")
    from umr_tools import to_valid
    try:
        return to_valid(balance_parens(text))
    except Exception:
        return "(z0 / empty-graph)", False

Writing umr_tokenizer.py


In [7]:
%%writefile train_amrbart.py
"""
train_amrbart.py - our own PyTorch fine-tuning loop for AMRBART on UMR (Sec 3.2:
"fine-tune text-to-AMR parsers on UMR data for 10 epochs with a learning rate of 4e-5").
Hyper-parameters the paper does not give are taken from AMRBART's own parsing script
(batch 16, label smoothing 0.1, warmup 200, weight decay 0.01, beam 5).
"""
import json
import math
import os
import random
import time

import torch
import torch.nn.functional as F

import umr_tools as U
from umr_tokenizer import decode_graph

CHECKPOINT = "xfbai/AMRBART-large-finetuned-AMR3.0-AMRParsing-v2"


# ---------------------------------------------------------------- data
def encode_sentence(tok, sent, max_src=400):
    # "unified input" format the AMRBART parser was trained with: text <AMR> <mask> </AMR>
    ids = tok(sent, truncation=True, max_length=max_src)["input_ids"][: max_src - 3]
    return ids + [tok.amr_bos_token_id, tok.mask_token_id, tok.amr_eos_token_id]


def build_features(recs, tok, max_src=400, max_tgt=1024):
    feats, truncated = [], 0
    for r in recs:
        lab = tok.tokenize_amr(U.linearize(r["umr"]))
        if len(lab) > max_tgt - 2:
            truncated += 1
        feats.append({"input_ids": encode_sentence(tok, r["sentence"], max_src),
                      "labels": lab[: max_tgt - 2] + [tok.amr_eos_token_id]})
    return feats, truncated


def batches(feats, bs, shuffle, seed=0):
    """Length-bucketed batches: shuffle, then sort inside chunks of 50 batches (faster, less padding)."""
    idx = list(range(len(feats)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    chunk = bs * 50
    out = []
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c:c + chunk], key=lambda i: len(feats[i]["labels"]))
        out += [part[i:i + bs] for i in range(0, len(part), bs)]
    if shuffle:
        random.Random(seed + 1).shuffle(out)
    return out


def collate(items, tok, device):
    pad = tok.pad_token_id
    L = max(len(x["input_ids"]) for x in items)
    T = max(len(x["labels"]) for x in items)
    inp = torch.full((len(items), L), pad)
    lab = torch.full((len(items), T), -100)
    for i, x in enumerate(items):
        inp[i, :len(x["input_ids"])] = torch.tensor(x["input_ids"])
        lab[i, :len(x["labels"])] = torch.tensor(x["labels"])
    dec = torch.full_like(lab, pad)
    dec[:, 0] = tok.amr_bos_token_id                       # decoder starts with <AMR>
    dec[:, 1:] = lab[:, :-1].masked_fill(lab[:, :-1] == -100, pad)
    return {"input_ids": inp.to(device), "attention_mask": (inp != pad).long().to(device),
            "decoder_input_ids": dec.to(device), "labels": lab.to(device)}


# ---------------------------------------------------------------- generation
@torch.no_grad()
def predict(model, tok, sentences, num_beams=5, bs=32, max_len=1024, device="cuda"):
    """Sentences -> list of (penman_string, ok)."""
    model.eval()
    enc = [encode_sentence(tok, s) for s in sentences]
    order = sorted(range(len(enc)), key=lambda i: len(enc[i]))
    out = [None] * len(enc)
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        L = max(len(enc[i]) for i in idx)
        inp = torch.full((len(idx), L), tok.pad_token_id)
        for j, i in enumerate(idx):
            inp[j, :len(enc[i])] = torch.tensor(enc[i])
        inp = inp.to(device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
            gen = model.generate(
                input_ids=inp, attention_mask=(inp != tok.pad_token_id).long(),
                decoder_start_token_id=tok.amr_bos_token_id, eos_token_id=tok.amr_eos_token_id,
                pad_token_id=tok.pad_token_id, num_beams=num_beams, max_length=max_len,
                min_length=0, no_repeat_ngram_size=0, length_penalty=1.0, early_stopping=num_beams > 1,
                forced_bos_token_id=None, forced_eos_token_id=None)
        for j, i in enumerate(idx):
            out[i] = decode_graph(tok, gen[j].tolist())
    return out


# ---------------------------------------------------------------- training
def train(model, tok, train_recs, dev_recs, out_dir, epochs=10, lr=4e-5, bs=16,
          warmup=200, weight_decay=0.01, label_smoothing=0.1, grad_clip=1.0,
          dev_subset=1000, seed=42, device="cuda", log_every=100, select="random"):
    """Fine-tune; after each epoch score a fixed dev subset (greedy, SMATCH) and keep the best.
    select="random"  : Job 1 behaviour - SMATCH on `dev_subset` random dev sentences.
    select="balanced": Job 3 - mean of SMATCH on Minecraft and non-Minecraft dev samples
                       (up to `dev_subset` each), so model selection does not favour Minecraft.
    Saves `last/` (for resuming after a Colab disconnect) and `best/` in out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(seed)
    random.seed(seed)
    feats, trunc = build_features(train_recs, tok)
    print(f"train examples: {len(feats)}  (labels truncated: {trunc})")
    if select == "balanced":
        mc = [r for r in dev_recs if r["is_minecraft"]]
        tx = [r for r in dev_recs if not r["is_minecraft"]]
        dev_groups = {"mc": random.Random(seed).sample(mc, min(dev_subset, len(mc))),
                      "txt": random.Random(seed).sample(tx, min(dev_subset, len(tx)))}
    else:
        dev_groups = {"all": random.Random(seed).sample(dev_recs, min(dev_subset, len(dev_recs)))}
    print("dev samples:", {k: len(v) for k, v in dev_groups.items()})

    no_decay = ("bias", "layer_norm.weight", "layernorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    steps_per_epoch = math.ceil(len(feats) / bs)
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warmup, max(0.0, (total - s) / max(1, total - warmup))))

    state_path = os.path.join(out_dir, "last", "train_state.pt")
    log_path = os.path.join(out_dir, "log.jsonl")
    start_epoch, best = 0, -1.0
    if os.path.exists(state_path):                         # resume
        st = torch.load(state_path, map_location="cpu")
        model.load_state_dict(torch.load(os.path.join(out_dir, "last", "model.pt"), map_location="cpu"))
        opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"])
        start_epoch, best = st["epoch"] + 1, st["best"]
        print(f"resumed after epoch {st['epoch'] + 1}, best dev score so far {best:.2f}")
    model.to(device)

    for ep in range(start_epoch, epochs):
        model.train()
        t0, run = time.time(), 0.0
        for k, b in enumerate(batches(feats, bs, shuffle=True, seed=seed + ep)):
            batch = collate([feats[i] for i in b], tok, device)
            labels = batch.pop("labels")
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                logits = model(**batch).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=-100, label_smoothing=label_smoothing)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            run += loss.item()
            if (k + 1) % log_every == 0:
                print(f"epoch {ep + 1} step {k + 1}/{steps_per_epoch} loss {run / log_every:.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} ({time.time() - t0:.0f}s)")
                run = 0.0

        # ---- dev check (greedy, fast) ----
        rec, scores, invalid = {"epoch": ep + 1}, [], 0
        for name, dev_sample in dev_groups.items():
            preds = predict(model, tok, [r["sentence"] for r in dev_sample], num_beams=1, device=device)
            tmp = os.path.join(out_dir, "dev_tmp")
            os.makedirs(tmp, exist_ok=True)
            U.write_amr_style([r["sentence"] for r in dev_sample], [U.to_valid(p)[0] for p, _ in preds],
                              os.path.join(tmp, "p.txt"))
            U.write_amr_style([r["sentence"] for r in dev_sample], [r["umr"] for r in dev_sample],
                              os.path.join(tmp, "g.txt"))
            sc = U.score_smatch(os.path.join(tmp, "p.txt"), os.path.join(tmp, "g.txt"))
            rec[f"dev_smatch_{name}"] = round(sc, 2)
            scores.append(sc)
            invalid += sum(not ok for _, ok in preds)
        sm = sum(scores) / len(scores)
        rec.update({"dev_select": round(sm, 2), "dev_invalid": invalid,
                    "minutes": round((time.time() - t0) / 60, 1)})
        print(rec)
        with open(log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")

        if sm > best:
            best = sm
            model.save_pretrained(os.path.join(out_dir, "best"))
            print(f"  new best -> saved {out_dir}/best")
        os.makedirs(os.path.join(out_dir, "last"), exist_ok=True)
        torch.save(model.state_dict(), os.path.join(out_dir, "last", "model.pt"))
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep, "best": best},
                   state_path)
    return best

Writing train_amrbart.py


In [8]:
%%writefile job3_tools.py
"""
job3_tools.py - Job 3: fixing the non-Minecraft weakness of the fine-tuned AMRBART parser.
Data re-weighting + paired bootstrap significance test.
"""
import random

import umr_tools as U


def upsample_text(recs, factor=4, seed=42):
    """Repeat non-Minecraft sentences `factor` times in total (Minecraft kept once), shuffled."""
    out = [r for r in recs if r["is_minecraft"]]
    txt = [r for r in recs if not r["is_minecraft"]]
    out += txt * factor
    random.Random(seed).shuffle(out)
    return out


def share_text(recs):
    return sum(not r["is_minecraft"] for r in recs) / max(1, len(recs))


# ---------------------------------------------------------------- per-sentence SMATCH counts
def smatch_counts(pred, gold):
    """(matched, predicted, gold) triple counts for one sentence, as SMATCH computes them.
    Summing these over sentences and taking F1 gives corpus SMATCH (micro), which is
    what the bootstrap resamples."""
    import smatch
    p = " ".join(U.to_valid(pred)[0].split())
    g = " ".join(gold.split())
    smatch.match_triple_dict.clear()
    m, t, gg = smatch.get_amr_match(p, g)
    smatch.match_triple_dict.clear()
    return m, t, gg


def f1(counts):
    m = sum(c[0] for c in counts); t = sum(c[1] for c in counts); g = sum(c[2] for c in counts)
    p, r = (m / t if t else 0.0), (m / g if g else 0.0)
    return 100 * 2 * p * r / (p + r) if p + r else 0.0


def paired_bootstrap(base_counts, new_counts, n=1000, seed=42):
    """Resample the same test sentences for both systems; returns
    (observed gain, 95% CI low, 95% CI high, share of resamples where the new system wins)."""
    assert len(base_counts) == len(new_counts)
    rng = random.Random(seed)
    N = len(base_counts)
    obs = f1(new_counts) - f1(base_counts)
    gains = []
    for _ in range(n):
        idx = [rng.randrange(N) for _ in range(N)]
        gains.append(f1([new_counts[i] for i in idx]) - f1([base_counts[i] for i in idx]))
    gains.sort()
    return round(obs, 2), round(gains[int(0.025 * n)], 2), round(gains[int(0.975 * n) - 1], 2), \
        round(sum(g > 0 for g in gains) / n, 3)

Writing job3_tools.py


In [9]:
!python setup_amrbart_tok.py AMRBART

amrbart_tok ready: ['__init__.py', 'additional-tokens.json', 'constant.py', 'penman_interface.py', 'postprocessing.py', 'tokenization_bart.py']


# Part 1 — Data and the Job 1 baseline

In [10]:
import json, gc, torch, pandas as pd
import umr_tools as U, train_amrbart as T, job3_tools as J
from umr_tokenizer import UMRBartTokenizer
from transformers import BartForConditionalGeneration
A = {k: U.load_jsonl(f'{DRIVE}/data/A/{k}.jsonl') for k in ('train', 'dev', 'test')}
B = {k: U.load_jsonl(f'{DRIVE}/data/B/{k}.jsonl') for k in ('train', 'dev', 'test')}
for n, S in (('A', A), ('B', B)):
    print(n, {k: f"{len(v)} ({J.share_text(v):.0%} non-MC)" for k, v in S.items()})
tok = UMRBartTokenizer.from_pretrained(T.CHECKPOINT)

def load_parser(path=T.CHECKPOINT):
    m = BartForConditionalGeneration.from_pretrained(path)
    assert len(tok.encoder) <= m.config.vocab_size
    return m.to('cuda')

def free(m):
    del m; gc.collect(); torch.cuda.empty_cache()

A {'train': '21460 (16% non-MC)', 'dev': '5897 (14% non-MC)', 'test': '2941 (16% non-MC)'}
B {'train': '24214 (16% non-MC)', 'dev': '1350 (22% non-MC)', 'test': '1358 (26% non-MC)'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AMRBartTokenizer'. 
The class this function is called from is 'UMRBartTokenizer'.


Added 0 AMR tokens


### 1.1 Baseline = your Job 1 AMRBART (no retraining)
Its saved test predictions give per-sentence SMATCH counts, so every new run is compared on **exactly the same sentences**.

In [11]:
def counts_of(preds, test):
    return [J.smatch_counts(p, x['umr']) for p, x in zip(preds, test)]

base = {}
for split, S in (('A', A), ('B', B)):
    saved = U.load_jsonl(f'{DRIVE}/runs/amrbart_{split}/test_predictions.jsonl')
    assert [x['sentence'] for x in saved] == [x['sentence'] for x in S['test']], 'test order changed'
    base[split] = {'scores': json.load(open(f'{DRIVE}/runs/amrbart_{split}/eval/test_scores.json')),
                   'counts': counts_of([x['pred'] for x in saved], S['test'])}
    print(split, 'baseline SMATCH from counts:', round(J.f1(base[split]['counts']), 2),
          '| reported:', base[split]['scores']['all']['SMATCH'])

A baseline SMATCH from counts: 91.15 | reported: 91.14
B baseline SMATCH from counts: 86.18 | reported: 86.19


### 1.2 Helper: test a run and compare with the baseline
Beam 5 test decoding (same as Job 1). Bootstrap reports the SMATCH **gain** over Job 1's AMRBART with a 95% confidence interval for all / Minecraft / non-Minecraft. A gain is reliable if the whole interval is above 0.

In [12]:
def test_run(run_dir, split, S):
    model = load_parser(f'{run_dir}/best')
    t = S['test']
    preds = T.predict(model, tok, [x['sentence'] for x in t], num_beams=5)
    free(model)
    U.save_jsonl([{**x, 'pred': g, 'valid': ok} for x, (g, ok) in zip(t, preds)], f'{run_dir}/test_predictions.jsonl')
    res = U.evaluate_all([x['sentence'] for x in t], [g for g, _ in preds], [x['umr'] for x in t],
                         [x['is_minecraft'] for x in t], f'{run_dir}/eval')
    cnt = counts_of([g for g, _ in preds], t)
    boot = {}
    for sub, keep in (('all', lambda x: True), ('minecraft', lambda x: x['is_minecraft']),
                      ('non_minecraft', lambda x: not x['is_minecraft'])):
        idx = [i for i, x in enumerate(t) if keep(x)]
        g, lo, hi, p = J.paired_bootstrap([base[split]['counts'][i] for i in idx], [cnt[i] for i in idx])
        boot[sub] = {'SMATCH gain': g, '95% CI': f'[{lo}, {hi}]', 'P(gain>0)': p}
    json.dump(boot, open(f'{run_dir}/eval/bootstrap_vs_amrbart.json', 'w'), indent=2)
    print('invalid predictions:', res['invalid_predictions'])
    display(pd.DataFrame({s: {k: res[s][k] for k in ('AnCast', 'SMATCH', 'SMATCH++')} for s in ('all', 'minecraft', 'non_minecraft')}).T)
    display(pd.DataFrame(boot).T)
    return res, boot

def best_dev(run_dir):
    """Best balanced dev score of a run (used to pick the winner - never the test set)."""
    rows = [json.loads(l) for l in open(f'{run_dir}/log.jsonl')]
    return max(rows, key=lambda r: r['dev_select'])

# Part 2 — Three fixes on split A

### R1 · Upsample non-Minecraft ×4 🔧
Same recipe as Job 1 (10 epochs, lr 4e-5, batch 16, from the AMRBART AMR parser) — **only the data mix changes**. ×4 lifts normal English from 16% to ~43% of training: the paper's §4.2 explanation is under-representation, so this tests it directly. Not ×5+: Minecraft would become the minority and could drop.

Model selection: `select="balanced"` 🔧 — mean of dev SMATCH on 1,000 Minecraft + up to 1,000 non-Minecraft dev sentences (Job 1's random 1,000 were 84% Minecraft and would reward the wrong thing). Resumes after a disconnect like Job 1.

In [13]:
R1 = f'{DRIVE}/runs/j3_upsample_A'
train_up = J.upsample_text(A['train'], factor=4)
print(f'train: {len(train_up)} examples, {J.share_text(train_up):.0%} non-Minecraft')
model = load_parser()
T.train(model, tok, train_up, A['dev'], R1, epochs=10, lr=4e-5, bs=16, select='balanced')
free(model)
res_R1, boot_R1 = test_run(R1, 'A', A)

train: 31705 examples, 43% non-Minecraft


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

train examples: 31705  (labels truncated: 0)
dev samples: {'mc': 1000, 'txt': 815}
resumed after epoch 10, best dev score so far 86.21
invalid predictions: 0


,AnCast,SMATCH,SMATCH++
all,86.74,91.00,91.00
minecraft,92.45,94.75,94.75
non_minecraft,66.90,77.57,77.54


,SMATCH gain,95% CI,P(gain>0)
all,-0.15,"[-0.43, 0.11]",0.14
minecraft,-0.1,"[-0.3, 0.11]",0.185
non_minecraft,-0.41,"[-1.48, 0.61]",0.234


### R2 · Two-stage: Job 1 model → extra training on non-Minecraft only 🔧
Starts from your Job 1 model (already good at Minecraft) and specialises on normal English: 3 epochs, **lr 1e-5** (¼ of Job 1, to avoid forgetting Minecraft), 20 warm-up steps (~3.4k sentences → ~215 steps/epoch). Balanced selection keeps the epoch that is best on **both** kinds of sentence — if Minecraft drops, that epoch won't win.

In [14]:
R2 = f'{DRIVE}/runs/j3_twostage_A'
txt_A = [r for r in A['train'] if not r['is_minecraft']]
print('stage-2 train:', len(txt_A))
model = load_parser(f'{DRIVE}/runs/amrbart_A/best')
T.train(model, tok, txt_A, A['dev'], R2, epochs=3, lr=1e-5, bs=16, warmup=20, select='balanced')
free(model)
res_R2, boot_R2 = test_run(R2, 'A', A)

stage-2 train: 3415
train examples: 3415  (labels truncated: 0)
dev samples: {'mc': 1000, 'txt': 815}
resumed after epoch 3, best dev score so far 86.91
invalid predictions: 0


,AnCast,SMATCH,SMATCH++
all,87.00,91.25,91.25
minecraft,92.70,94.91,94.92
non_minecraft,67.22,78.13,78.13


,SMATCH gain,95% CI,P(gain>0)
all,0.11,"[-0.05, 0.25]",0.897
minecraft,0.07,"[-0.06, 0.19]",0.851
non_minecraft,0.19,"[-0.36, 0.73]",0.756


### R3 · Combined: R1's model → the R2 second stage

In [15]:
R3 = f'{DRIVE}/runs/j3_combined_A'
model = load_parser(f'{R1}/best')
T.train(model, tok, txt_A, A['dev'], R3, epochs=3, lr=1e-5, bs=16, warmup=20, select='balanced')
free(model)
res_R3, boot_R3 = test_run(R3, 'A', A)

train examples: 3415  (labels truncated: 0)
dev samples: {'mc': 1000, 'txt': 815}
resumed after epoch 3, best dev score so far 85.91
invalid predictions: 0


,AnCast,SMATCH,SMATCH++
all,86.66,91.01,91.01
minecraft,92.48,94.77,94.77
non_minecraft,66.42,77.52,77.51


,SMATCH gain,95% CI,P(gain>0)
all,-0.14,"[-0.41, 0.13]",0.146
minecraft,-0.08,"[-0.28, 0.13]",0.231
non_minecraft,-0.45,"[-1.64, 0.66]",0.226


# Part 3 — Pick the winner on dev, confirm on clean split B

The winner is the run with the highest **balanced dev score** — chosen before looking at test, so the split-B result is an honest confirmation.
Split B baseline = your Job 1 `amrbart_B`; the winner's recipe is repeated exactly on B.

In [16]:
devs = {name: best_dev(d) for name, d in (('R1 upsample', R1), ('R2 two-stage', R2), ('R3 combined', R3))}
display(pd.DataFrame(devs).T)
WIN = max(devs, key=lambda k: devs[k]['dev_select'])
print('winner on dev:', WIN)

,epoch,dev_smatch_mc,dev_smatch_txt,dev_select,dev_invalid,minutes
R1 upsample,10.0,95.11,77.31,86.21,18.0,5.9
R2 two-stage,2.0,94.99,78.83,86.91,5.0,1.9
R3 combined,1.0,95.04,76.78,85.91,20.0,1.9


winner on dev: R2 two-stage


In [18]:
WB = f'{DRIVE}/runs/j3_winner_B'
import shutil; shutil.rmtree(WB, ignore_errors=True)
txt_B = [r for r in B['train'] if not r['is_minecraft']]
if WIN.startswith('R1'):
    model = load_parser()
    T.train(model, tok, J.upsample_text(B['train'], 4), B['dev'], WB, epochs=10, lr=4e-5, bs=16, select='balanced')
elif WIN.startswith('R2'):
    model = load_parser(f'{DRIVE}/runs/amrbart_B/best')
    T.train(model, tok, txt_B, B['dev'], WB, epochs=3, lr=1e-5, bs=16, warmup=20, select='balanced')
else:  # R3: upsample on B, then the second stage
    WB1 = f'{DRIVE}/runs/j3_upsample_B'
    model = load_parser()
    T.train(model, tok, J.upsample_text(B['train'], 4), B['dev'], WB1, epochs=10, lr=4e-5, bs=16, select='balanced')
    free(model)
    model = load_parser(f'{WB1}/best')
    T.train(model, tok, txt_B, B['dev'], WB, epochs=3, lr=1e-5, bs=16, warmup=20, select='balanced')
free(model)
res_WB, boot_WB = test_run(WB, 'B', B)

train examples: 3841  (labels truncated: 0)
dev samples: {'mc': 1000, 'txt': 298}
epoch 1 step 100/241 loss 1.4638 lr 8.86e-06 (13s)
epoch 1 step 200/241 loss 1.4689 lr 7.44e-06 (26s)
{'epoch': 1, 'dev_smatch_mc': 89.79, 'dev_smatch_txt': 78.42, 'dev_select': 84.1, 'dev_invalid': 8, 'minutes': 1.6}
  new best -> saved /content/drive/MyDrive/umr_job1/runs/j3_winner_B/best
epoch 2 step 100/241 loss 1.4637 lr 5.43e-06 (13s)
epoch 2 step 200/241 loss 1.4638 lr 4.01e-06 (26s)
{'epoch': 2, 'dev_smatch_mc': 89.77, 'dev_smatch_txt': 78.19, 'dev_select': 83.98, 'dev_invalid': 8, 'minutes': 1.7}
epoch 3 step 100/241 loss 1.4604 lr 2.01e-06 (13s)
epoch 3 step 200/241 loss 1.4608 lr 5.83e-07 (26s)
{'epoch': 3, 'dev_smatch_mc': 89.71, 'dev_smatch_txt': 78.4, 'dev_select': 84.05, 'dev_invalid': 9, 'minutes': 1.8}
invalid predictions: 0


,AnCast,SMATCH,SMATCH++
all,78.97,86.03,86.06
minecraft,84.03,89.23,89.23
non_minecraft,67.66,78.75,78.79


,SMATCH gain,95% CI,P(gain>0)
all,-0.13,"[-0.36, 0.1]",0.132
minecraft,-0.04,"[-0.28, 0.2]",0.36
non_minecraft,-0.32,"[-0.78, 0.13]",0.083


# Part 4 — Final table

In [19]:
rows = {}
def add(name, res):
    for s in ('all', 'minecraft', 'non_minecraft'):
        rows[f'{name} – {s}'] = {k: res[s][k] for k in ('n', 'AnCast', 'SMATCH', 'SMATCH++')}
rows['Paper AMRBART (Table 5) – all'] = {'AnCast': 81.70, 'SMATCH': 86.24, 'SMATCH++': 88.70}
add('A · Job 1 AMRBART', base['A']['scores'])
add('A · R1 upsample', res_R1); add('A · R2 two-stage', res_R2); add('A · R3 combined', res_R3)
add('B · Job 1 AMRBART', base['B']['scores'])
add(f'B · {WIN}', res_WB)
final = pd.DataFrame(rows).T
final.to_csv(f'{DRIVE}/job3_results.csv')
print(final.round(2).to_string())

brow = {}
for name, b in (('A · R1 upsample', boot_R1), ('A · R2 two-stage', boot_R2),
                ('A · R3 combined', boot_R3), (f'B · {WIN}', boot_WB)):
    for s, v in b.items():
        brow[f'{name} – {s}'] = v
bt = pd.DataFrame(brow).T
bt.to_csv(f'{DRIVE}/job3_bootstrap.csv')
print('\nSMATCH gain over Job 1 AMRBART (paired bootstrap, 1,000 resamples):')
print(bt.to_string())

                                   AnCast  SMATCH  SMATCH++       n
Paper AMRBART (Table 5) – all       81.70   86.24     88.70     NaN
A · Job 1 AMRBART – all             86.85   91.14     91.15  2941.0
A · Job 1 AMRBART – minecraft       92.64   94.84     94.85  2478.0
A · Job 1 AMRBART – non_minecraft   66.82   77.97     77.95   463.0
A · R1 upsample – all               86.74   91.00     91.00  2941.0
A · R1 upsample – minecraft         92.45   94.75     94.75  2478.0
A · R1 upsample – non_minecraft     66.90   77.57     77.54   463.0
A · R2 two-stage – all              87.00   91.25     91.25  2941.0
A · R2 two-stage – minecraft        92.70   94.91     94.92  2478.0
A · R2 two-stage – non_minecraft    67.22   78.13     78.13   463.0
A · R3 combined – all               86.66   91.01     91.01  2941.0
A · R3 combined – minecraft         92.48   94.77     94.77  2478.0
A · R3 combined – non_minecraft     66.42   77.52     77.51   463.0
B · Job 1 AMRBART – all             79.35   86.1

### Reading the result
- **Non-Minecraft gain with the whole CI above 0** on split A **and** B → a real fix for the paper's stated limitation.
- Check the Minecraft rows: a small drop is an acceptable trade if non-Minecraft gains more; say so honestly.
- R2/R3 train longer than the baseline (extra epochs) — mention it.
- Caveats for the write-up: one seed per run; winner chosen on dev; bootstrap covers test-sample noise, not training noise.